In [5]:
import json
import time
import os 
from dotenv import load_dotenv
from datetime import date
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from seleniumwire import webdriver
import os

load_dotenv(override=True)

def get_selenium_wire_options():
    user = os.getenv("WEBSHARE_USERNAME")
    pw = os.getenv("WEBSHARE_PASSWORD")
    host = os.getenv("WEBSHARE_HOST")
    port = os.getenv("WEBSHARE_PORT")
    
    proxy_url = f"http://{user}:{pw}@{host}:{port}"
    
    return {
        'proxy': {
            'http': proxy_url,
            'https': proxy_url,
            'no_proxy': 'localhost,127.0.0.1'
        }
    }

print(" Cell 1: Libraries loaded and Webshare functions defined.")

 Cell 1: Libraries loaded and Webshare functions defined.


In [6]:

chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

sw_options = get_selenium_wire_options()

print("Attempting to connect to Webshare (Headless Mode)...")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    seleniumwire_options=sw_options,
    options=chrome_options  
)

try:
    driver.get("https://ipv4.icanhazip.com")
    proxy_ip = driver.find_element("tag name", "body").text.strip()
    print(f"SUCCESS! You are connected via Webshare.")
    print(f"Your Proxy IP is: {proxy_ip}")

except Exception as e:
    print(f"Connection failed: {e}")

finally:
    print("Closing the browser...")
    driver.quit()

Attempting to connect to Webshare (Headless Mode)...
SUCCESS! You are connected via Webshare.
Your Proxy IP is: 31.59.20.176
Closing the browser...


In [7]:
chrome_options = Options()
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--headless")

output_dir = os.path.join("smarter-scraping", "dataset")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f" Created directory: {output_dir}")

today = date.today()
max_pages = 5 
url_template = "https://finance.yahoo.com/research-hub/screener/sec-ind_sec-largest-equities_technology/?start={start}&count=25"

all_extracted_data = []

print(f" Starting scrape and extraction for {today}...")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

try:
    for page in range(max_pages):
        start_index = page * 25
        target_url = url_template.format(start=start_index)
        print(f" Processing Page {page + 1}: {target_url}")
        
        driver.get(target_url)
        
        try:
            # Wait for rows using the data-testid from your screenshot
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "tr[data-testid='data-table-v2-row']"))
            )
            time.sleep(2) 
        except Exception as e:
            print(f" Timeout on page {page + 1}. skipping...")
            continue

        soup = BeautifulSoup(driver.page_source, 'html.parser')
        # rows = soup.find_all("tr", attrs={"data-testid": "data-table-v2-row"})
        rows = soup.find_all("tr", attrs={"data-testid-row": True})
        print(f"Found {len(rows)} rows on the page.")
        
        
        for row in rows:
            try:
                def get_val(cell_tid):
                    cell = row.find("td", {"data-testid-cell": cell_tid})
                    if not cell:
                        return "N/A"

                    return cell.get_text(strip=True)

                name_cell = row.find("td", {"data-testid-cell": "companyshortname.raw"})
                full_name = "N/A"
                if name_cell and name_cell.find("div"):
                    full_name = name_cell.find("div").get("title", get_val("companyshortname.raw"))
                stock_entry = {
                    "ticker": get_val("ticker"),
                    "name": full_name,
                    "price": get_val("intradayprice"),
                    "change": get_val("intradaypricechange"),
                    "percent_change": get_val("percentchange"),
                    "market_cap": get_val("intradaymarketcap"),
                    "pe_ratio": get_val("peratio.lasttwelvemonths"),
                    "volume": get_val("dayvolume"),
                    "sector": get_val("sector"),
                    "region": get_val("region"),
                    "scraped_at": str(today)
                }

                if stock_entry["ticker"] != "N/A":
                    all_extracted_data.append(stock_entry)
                    
            except Exception as e:
                continue 

    print(f" Extraction Complete! Total stocks collected: {len(all_extracted_data)}")

    file_name = f"tech_market_data_{today}.json"
    full_path = os.path.join(output_dir, file_name)
    
    with open(full_path, "w", encoding="utf-8") as f:
        json.dump(all_extracted_data, f, indent=4)
        
    print(f" Clean data saved to: {full_path}")

except Exception as e:
    print(f" Scraper stopped: {e}")

finally:
    try:
        driver.quit()
        print(" Browser closed.")
    except:
        pass

 Starting scrape and extraction for 2026-04-06...
 Processing Page 1: https://finance.yahoo.com/research-hub/screener/sec-ind_sec-largest-equities_technology/?start=0&count=25
Found 25 rows on the page.
 Processing Page 2: https://finance.yahoo.com/research-hub/screener/sec-ind_sec-largest-equities_technology/?start=25&count=25
Found 25 rows on the page.
 Processing Page 3: https://finance.yahoo.com/research-hub/screener/sec-ind_sec-largest-equities_technology/?start=50&count=25
Found 25 rows on the page.
 Processing Page 4: https://finance.yahoo.com/research-hub/screener/sec-ind_sec-largest-equities_technology/?start=75&count=25
Found 25 rows on the page.
 Processing Page 5: https://finance.yahoo.com/research-hub/screener/sec-ind_sec-largest-equities_technology/?start=100&count=25
Found 25 rows on the page.
 Extraction Complete! Total stocks collected: 125
 Clean data saved to: smarter-scraping\dataset\tech_market_data_2026-04-06.json
 Browser closed.


In [8]:
import os
print(os.getcwd())

C:\Users\Lenovo Loq\OneDrive\Documents\webscrapping\Loq\dev\smarter-scraping\src
